# Gradient of Periodic Splines
The spline $f$ whose gradient is sought is shown as a <span style="color:#c20078">**fuchsia**</span> curve, with data samples at the integers represented with circles and stem lines. The sample at the origin, as well as its periodized replicates, is indicated by a red circle and stem line. The small black disks depict the knots of the spline. The gradient of $f$ is shown as thick curve in the <span style="background-color:#d8ebd6">pale green</span> color. If desired, the finite-difference approximation of the gradient can be shown in <span style="color:#7f7f7f">**gray**</span>.

It is instructive to realize that the knots of the spline and its gradient share their abscissa. This indicates that the delay associated to the gradient is a half-integer away from the delay associated to the spline $f$ because the degree of one of these two splines is even when the degree of the other one is odd, and reciprocally.

In [ ]:
# Load the required libraries
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
max_period = 15 # Maximal period
max_degree = 9 # Maximal spline degree
max_delay = 2.0 # Maximal absolute delay
plotpoints = 200 + 1 # Number of rendered samples

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Initial random periodic cubic splines with normal Gaussian coefficients
f = sk.PeriodicSpline1D.from_spline_coeff(rng.standard_normal(6), degree = 3)

# Plot
def update_plot (
    period = 6,
    degree = 1,
    delay = 0.0,
    finite_diff = False,
    m = 4
):
    global f

    # Update of the spline
    if f.period != period:
        f = sk.PeriodicSpline1D.from_spline_coeff(
            rng.standard_normal(period),
            degree = f.degree
        )
    f.degree = degree
    f.delay = delay

    # True gradient
    g = f.gradient()

    # Dynamic range
    image = {f.image(), g.image()}
    plotrange = sk.interval.Interval.enclosure(image)
    plotrange = sk.interval.Closed((
        plotrange.midpoint - 0.55 * plotrange.diameter,
        plotrange.midpoint + 0.55 * plotrange.diameter
    ))

    # Plot of the gradient
    subplot = plt.subplots()
    g.plot(
        subplot,
        plotrange = plotrange,
        plotpoints = 200 + 1,
        curve_fmt = "#d8ebd6",
        curve_lw = 7.0
    )

    # Finite differences
    if finite_diff:
        x = np.linspace(start = -1.0, stop = period + 1.0, num = plotpoints)
        h = 1.0 / m
        fdiff = 0.5 * m * np.array([f.at(x0 + h) - f.at(x0 - h) for x0 in x])
        plt.plot(x, fdiff, "-C7", linewidth = 0.75)
        m_int_slider_widget.disabled = False
    else:
        m_int_slider_widget.disabled = True

    # Plot of the spline
    f.plot(subplot, plotrange = plotrange, plotpoints = 200 + 1, curve_fmt = "-m")
    plt.show()

# Interaction
m_int_slider_widget = widgets.IntSlider(
    value = 4,
    min = 1,
    max = 10,
    description = "m:",
    disabled = True
)
widgets.interactive(
    update_plot,
    period = (1, max_period),
    degree = (1, max_degree),
    delay = (-max_delay, max_delay),
    finite_diff = widgets.Checkbox(
        value = False,
        description = "Finite differences, step 1/m"
    ),
    m = m_int_slider_widget
)
